#Importing all libreries


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

#Conecting google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Load Data

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Thesis/Survey/Survey_Responses.csv')

# 2. Variable Mapping (Mapping Google Form questions to Thesis Conceptual Framework)

In [ ]:
col_mapping = {
    'Which category best describes your current enrollment status and duration of residence at your university in Germany?': 'Nationality',
    'How often do you feel overwhelmed by your perceived workload and university expectations?': 'Academic_Stress',
    'Overall, how frequently do academic or environmental stressors disrupt your general sleep quality or sleep regularity?': 'Sleep_Disruption',
    'Overall, how often do you find your daily functioning or academic focus impaired by mental exhaustion or an inability to concentrate?': 'Mental_Exhaustion',
    'In general, how often do you intentionally avoid social contact with friends, family, or peers when feeling stressed?': 'Social_Withdrawal',
    'I would only trust the AI\'s screening result if it provided a clear explanation of why it flagged a risk.': 'XAI_Trust'
}
df.rename(columns=col_mapping, inplace=True)
df = df.dropna(subset=['Nationality', 'Academic_Stress'])

print("================================================================")
print(" STAGE 2: CORE SCREENING VARIABLES CORRELATION (COLLINEARITY)")
print("================================================================")
# Testing relationships between Independent Variables before model fusion
core_vars = ['Academic_Stress', 'Sleep_Disruption', 'Mental_Exhaustion', 'Social_Withdrawal']
correlation_matrix = df[core_vars].corr(method='pearson')
print(correlation_matrix.round(3))
print("\nNote: High correlation (>0.7) indicates potential multicollinearity between features.")

print("\n===========================================================")
print(" HYPOTHESIS 2 (H2): DEMOGRAPHIC VARIANCE TESTING (T-TEST)")
print("=============================================================")
# Splitting groups based on Nationality Moderator
german_students = df[df['Nationality'].str.contains('German', na=False)]['XAI_Trust']
intl_students = df[df['Nationality'].str.contains('International', na=False)]['XAI_Trust']

if len(german_students) > 0 and len(intl_students) > 0:
    # Perform Independent T-Test
    t_stat, p_val = stats.ttest_ind(german_students, intl_students, equal_var=False)

    print(f"German Students Mean XAI Trust: {german_students.mean():.2f} (n={len(german_students)})")
    print(f"Intl Students Mean XAI Trust: {intl_students.mean():.2f} (n={len(intl_students)})")
    print(f"\nT-Statistic: {t_stat:.3f}")
    print(f"P-Value: {p_val:.4f}")

    if p_val < 0.05:
        print("-> Result: Statistically significant difference found. (Supports H2)")
    else:
        print("-> Result: No statistically significant difference found. (Reject H2)")
else:
    print("Insufficient data in one or both demographic groups to perform T-test.")

 STAGE 2: CORE SCREENING VARIABLES CORRELATION (COLLINEARITY)
                   Academic_Stress  Sleep_Disruption  Mental_Exhaustion  \
Academic_Stress              1.000            -0.056              0.208   
Sleep_Disruption            -0.056             1.000              0.434   
Mental_Exhaustion            0.208             0.434              1.000   
Social_Withdrawal            0.043             0.061              0.416   

                   Social_Withdrawal  
Academic_Stress                0.043  
Sleep_Disruption               0.061  
Mental_Exhaustion              0.416  
Social_Withdrawal              1.000  

Note: High correlation (>0.7) indicates potential multicollinearity between features.

 HYPOTHESIS 2 (H2): DEMOGRAPHIC VARIANCE TESTING (T-TEST)
Insufficient data in one or both demographic groups to perform T-test.


## Summary of Analysis

This notebook performs an initial analysis of survey responses, focusing on preparing the data and conducting preliminary statistical tests.

**Key steps included:**

1.  **Library Imports**: Essential libraries such as `pandas` for data manipulation, `numpy` for numerical operations, and `scipy.stats` for statistical functions were imported.
2.  **Google Drive Connection**: The notebook connected to Google Drive to access the `Survey_Responses.csv` file.
3.  **Data Loading**: The survey data was loaded into a pandas DataFrame.
4.  **Variable Mapping**: Google Form questions were mapped to conceptual framework variables like 'Nationality', 'Academic_Stress', 'Sleep_Disruption', 'Mental_Exhaustion', and 'Social_Withdrawal'. Rows with missing values in 'Nationality' or 'Academic_Stress' were removed.
5.  **Collinearity Check (Stage 2)**: A Pearson correlation matrix was computed for core screening variables (`Academic_Stress`, `Sleep_Disruption`, `Mental_Exhaustion`, `Social_Withdrawal`) to identify potential multicollinearity (indicated by correlations > 0.7).
6.  **Hypothesis 2 (H2) - Demographic Variance Testing**: An independent t-test was attempted to compare 'XAI_Trust' levels between German and International students. The output indicated 'Insufficient data in one or both demographic groups' which means the t-test could not be performed due to lack of data for comparison.